# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



## 1. My rule and its reason code

My Refresh / Content Opportunity rule prioritizes pages that have not been updated for at least 91 days but still received at least 500 search impressions. Pages with more impressions and greater staleness receive a higher score.

- **Reason code:** `stale_but_visible`
- **Action label:** `review_for_refresh`

### Signal 1 — staleness

Staleness is used behind FlyRank's refresh flags. I compare visible pages updated within 90 days with pages not updated for at least 91 days.

**Verdict: CONFIRMED.** Among visible pages, the stale bucket has a moderately higher observed decline rate.

### Signal 2 — search volume

Volume is used behind quick-win prioritization. I compare decline rates across impression buckets among stale pages.

**Verdict: MIXED.** The decline rate does not increase consistently with volume. Therefore, volume represents possible impact, not proof of decline.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np
from IPython.display import display

local_data = Path("data/raw/content_refresh_anonymized.csv")
colab_data = Path(
    "flyrank-starter/data/raw/content_refresh_anonymized.csv"
)

if local_data.exists():
    DATA_PATH = local_data
elif colab_data.exists():
    DATA_PATH = colab_data
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/flyrank-bih/"
            "flyrank-ml-internship-starter.git",
            "flyrank-starter",
        ],
        check=True,
    )
    DATA_PATH = colab_data

df = pd.read_csv(DATA_PATH)

# Evaluation proxy only; never a rule input.
df["is_declining_proxy"] = (
    df["trend_direction"].eq("down").astype(int)
)

print("Dataset shape:", df.shape)
print("One row per content item:", df["content_id"].is_unique)


# SIGNAL 1 — STALENESS

visible = df.loc[df["impressions_90d"] >= 500].copy()

visible["staleness_bucket"] = pd.Categorical(
    np.where(
        visible["days_since_last_update"] >= 91,
        "91+ days",
        "0-90 days",
    ),
    categories=["0-90 days", "91+ days"],
    ordered=True,
)

staleness_table = (
    visible.groupby("staleness_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        decline_rate=("is_declining_proxy", "mean"),
        median_impressions=("impressions_90d", "median"),
    )
    .reset_index()
)

print("\nSignal 1 — staleness")
display(staleness_table.round(3))
print("Verdict: CONFIRMED")


# SIGNAL 2 — VOLUME

stale_pages = df.loc[
    df["days_since_last_update"] >= 91
].copy()

stale_pages["volume_bucket"] = pd.cut(
    stale_pages["impressions_90d"],
    bins=[-1, 499, 1999, 9999, np.inf],
    labels=[
        "<500",
        "500-1,999",
        "2,000-9,999",
        "10,000+",
    ],
)

volume_table = (
    stale_pages.groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        decline_rate=("is_declining_proxy", "mean"),
        median_days_stale=("days_since_last_update", "median"),
    )
    .reset_index()
)

print("\nSignal 2 — search volume")
display(volume_table.round(3))
print("Verdict: MIXED")

Dataset shape: (30000, 45)
One row per content item: True

Signal 1 — staleness


,staleness_bucket,n,decline_rate,median_impressions
0,0-90 days,10151,0.582,2688.0
1,91+ days,6575,0.616,3435.0


Verdict: CONFIRMED

Signal 2 — search volume


,volume_bucket,n,decline_rate,median_days_stale
0,<500,2770,0.590,104.0
1,"500-1,999",2275,0.656,104.0
2,"2,000-9,999",2666,0.639,104.0
3,"10,000+",1634,0.524,104.0


Verdict: MIXED


The rule keeps pages with at least 500 impressions and at least 91 days since their recorded update.

The transparent score is:

`log(1 + impressions) × staleness weight`

The staleness weight increases gradually from 1 to 2 and is capped after 270 days. The score prioritizes human review; it does not prove that refreshing a page will improve performance.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
RULE_INPUTS = [
    "impressions_90d",
    "days_since_last_update",
]

eligible = (
    (df["impressions_90d"] >= 500)
    & (df["days_since_last_update"] >= 91)
)

queue = df.loc[eligible].copy()

staleness_weight = 1 + (
    queue["days_since_last_update"]
    .sub(90)
    .clip(lower=0, upper=180)
    / 180
)

queue["baseline_action_score"] = (
    np.log1p(queue["impressions_90d"])
    * staleness_weight
)

# Exactly one reason code and one action label.
queue["reason_code"] = "stale_but_visible"
queue["action_label"] = "review_for_refresh"

queue = (
    queue.sort_values(
        ["baseline_action_score", "impressions_90d"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

# Evaluation only.
base_rate = df["is_declining_proxy"].mean()
precision_at_10 = queue.head(10)["is_declining_proxy"].mean()
precision_at_50 = queue.head(50)["is_declining_proxy"].mean()

print("Ranked candidates:", len(queue))
print(f"Observed proxy base rate: {base_rate:.3f}")
print(f"Baseline Precision@10: {precision_at_10:.3f}")
print(f"Baseline Precision@50: {precision_at_50:.3f}")

export_columns = [
    "rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

queue_export = queue[export_columns].copy()

OUTPUT_PATH = Path(
    "work/outputs/baseline_action_score.csv"
)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

queue_export.to_csv(OUTPUT_PATH, index=False)

print("\nCSV written to:", OUTPUT_PATH)
display(queue_export.head(10))

Ranked candidates: 6575
Observed proxy base rate: 0.542
Baseline Precision@10: 0.900
Baseline Precision@50: 0.560

CSV written to: work/outputs/baseline_action_score.csv


,rank,content_id,baseline_action_score,reason_code,action_label,impressions_90d,days_since_last_update,avg_position,ctr
0,1,content_cf56e2e2e282,17.402414,stale_but_visible,review_for_refresh,61678,194,19.7,0.15
1,2,content_7368877ea310,17.344949,stale_but_visible,review_for_refresh,59472,194,24.8,0.13
2,3,content_1bfaa38ff26c,16.022126,stale_but_visible,review_for_refresh,25715,194,22.2,0.23
3,4,content_0a91db491d14,14.929066,stale_but_visible,review_for_refresh,13299,193,10.5,0.49
4,5,content_5fe46e04994d,14.180518,stale_but_visible,review_for_refresh,517715,104,4.2,0.14
5,6,content_5feee3994adb,14.142481,stale_but_visible,review_for_refresh,7812,194,39.0,0.01
6,7,content_c2d929d83eaa,14.040721,stale_but_visible,review_for_refresh,7558,193,17.9,0.20
7,8,content_2dba2b1f9536,14.013597,stale_but_visible,review_for_refresh,443434,104,27.9,0.21
8,9,content_2c2606c5d176,13.750539,stale_but_visible,review_for_refresh,347399,104,4.2,0.53
9,10,content_7f116ae1f6f5,13.723423,stale_but_visible,review_for_refresh,954,301,9.0,0.42


## 3. Top-10 review

Each row below includes the proposed action, the rule's single reason code, a confidence note, and one condition that could make the recommendation wrong. These are review candidates, not automatic refresh decisions.

In [3]:
def wrong_reason(row):
    if row["impressions_90d"] < 2000:
        return (
            "The traffic volume is modest, so the priority "
            "may be unstable."
        )

    if row["avg_position"] > 20:
        return (
            "Low ranking may reflect authority or competition; "
            "refreshing alone may not help."
        )

    if row["avg_position"] <= 10 and row["ctr"] >= 0.5:
        return (
            "The page may already capture clicks efficiently "
            "despite being old."
        )

    return (
        "Demand may be temporary, or the update date may not "
        "represent a meaningful content change."
    )


reviews = []

for _, row in queue.head(10).iterrows():
    confidence = (
        "HIGH"
        if row["impressions_90d"] >= 10000
        and row["avg_position"] <= 20
        else "MEDIUM"
    )

    reviews.append(
        {
            "rank": int(row["rank"]),
            "content_id": row["content_id"],
            "action": row["action_label"],
            "reason_code": row["reason_code"],
            "confidence": confidence,
            "why": (
                f"{int(row['days_since_last_update'])} days "
                f"since update and "
                f"{int(row['impressions_90d']):,} impressions."
            ),
            "what_would_make_it_wrong": wrong_reason(row),
        }
    )

top_10_review = pd.DataFrame(reviews)

pd.set_option("display.max_colwidth", None)
display(top_10_review)

,rank,content_id,action,reason_code,confidence,why,what_would_make_it_wrong
0,1,content_cf56e2e2e282,review_for_refresh,stale_but_visible,HIGH,"194 days since update and 61,678 impressions.","Demand may be temporary, or the update date may not represent a meaningful content change."
1,2,content_7368877ea310,review_for_refresh,stale_but_visible,MEDIUM,"194 days since update and 59,472 impressions.",Low ranking may reflect authority or competition; refreshing alone may not help.
2,3,content_1bfaa38ff26c,review_for_refresh,stale_but_visible,MEDIUM,"194 days since update and 25,715 impressions.",Low ranking may reflect authority or competition; refreshing alone may not help.
3,4,content_0a91db491d14,review_for_refresh,stale_but_visible,HIGH,"193 days since update and 13,299 impressions.","Demand may be temporary, or the update date may not represent a meaningful content change."
4,5,content_5fe46e04994d,review_for_refresh,stale_but_visible,HIGH,"104 days since update and 517,715 impressions.","Demand may be temporary, or the update date may not represent a meaningful content change."
5,6,content_5feee3994adb,review_for_refresh,stale_but_visible,MEDIUM,"194 days since update and 7,812 impressions.",Low ranking may reflect authority or competition; refreshing alone may not help.
6,7,content_c2d929d83eaa,review_for_refresh,stale_but_visible,MEDIUM,"193 days since update and 7,558 impressions.","Demand may be temporary, or the update date may not represent a meaningful content change."
7,8,content_2dba2b1f9536,review_for_refresh,stale_but_visible,MEDIUM,"104 days since update and 443,434 impressions.",Low ranking may reflect authority or competition; refreshing alone may not help.
8,9,content_2c2606c5d176,review_for_refresh,stale_but_visible,HIGH,"104 days since update and 347,399 impressions.",The page may already capture clicks efficiently despite being old.
9,10,content_7f116ae1f6f5,review_for_refresh,stale_but_visible,MEDIUM,301 days since update and 954 impressions.,"The traffic volume is modest, so the priority may be unstable."


The rule can produce weak picks because an old, visible page is not necessarily a refresh opportunity. Demand may be temporary, the recorded update date may be incomplete, or poor ranking may be caused by competition rather than stale content.

`trend_direction` and `is_declining_proxy` are used only after ranking to evaluate the baseline. They are never score inputs. The score uses no future-window fields, product flags, identifiers, or label-derived inputs.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
weak_picks = queue.head(10).loc[
    queue.head(10)["is_declining_proxy"] == 0,
    [
        "rank",
        "content_id",
        "baseline_action_score",
        "trend_direction",
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr",
    ],
].copy()

print("Weak picks among the top 10:", len(weak_picks))

if len(weak_picks) > 0:
    display(weak_picks)
else:
    print("No proxy-negative rows appeared in the top 10.")


FORBIDDEN_SCORE_INPUTS = {
    "trend_direction",
    "trend_pct",
    "is_declining_proxy",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
}

assert set(RULE_INPUTS).isdisjoint(
    FORBIDDEN_SCORE_INPUTS
)
assert queue_export["reason_code"].nunique() == 1
assert queue_export["action_label"].nunique() == 1
assert OUTPUT_PATH.exists()
assert len(top_10_review) == 10

print("\nLeakage check passed.")
print("Rule inputs:", RULE_INPUTS)
print("Reason code:", queue_export["reason_code"].unique())
print("Action label:", queue_export["action_label"].unique())
print("CSV exists:", OUTPUT_PATH.exists())

Weak picks among the top 10: 1


,rank,content_id,baseline_action_score,trend_direction,impressions_90d,days_since_last_update,avg_position,ctr
7,8,content_2dba2b1f9536,14.013597,stable,443434,104,27.9,0.21



Leakage check passed.
Rule inputs: ['impressions_90d', 'days_since_last_update']
Reason code: ['stale_but_visible']
Action label: ['review_for_refresh']
CSV exists: True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.